# Investment Research Agent
**AAI-520 Final Project — University of San Diego**

Team: Viviana Garzon · Obinna Edeh · Dina Othman

---

## Project Goal
Build a multi-agent AI system that researches any stock ticker and generates a professional investment report automatically — pulling live price data, financials, and news, then analyzing and refining the output using three workflow patterns:

| Workflow | What it does |
|---|---|
| **Prompt Chaining** | Ingest news → clean → classify → extract facts → summarize |
| **Routing** | Classify content and dispatch to the right specialist agent |
| **Evaluator-Optimizer** | Score a draft, give feedback, rewrite — repeat until quality passes |


In [ ]:
#1. Initial Setup
from pathlib import Path
import sys

# Add repo root to path when running from notebooks/
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root))

from dotenv import load_dotenv
load_dotenv(repo_root / ".env")

print("Setup complete. Repo root:", repo_root)

In [ ]:
#1. YFinance Tool Usage
from tools.yfinance_tool import get_stock_data, get_financials

TICKER = "AAPL"

market = get_stock_data(TICKER)
print(f"Ticker:        {market['ticker']}")
print(f"Current price: ${market['current_price']:,.2f}")
print(f"Change today:  {market['change_pct']:+.2f}%")
print(f"52W range:     ${market['week_52_low']} – ${market['week_52_high']}")
print(f"MA50:          {market['ma_50']}")
print(f"History rows:  {len(market['history'])}")

In [ ]:
fin = get_financials(TICKER)
print(f"Sector:        {fin['sector']}")
print(f"Revenue:       {fin['revenue']}")
print(f"Net income:    {fin['net_income']}")
print(f"P/E ratio:     {fin['pe_ratio']}")

In [ ]:
from tools.news_tool import fetch_news

articles = fetch_news(TICKER, lookback_days=7, max_articles=5)
print(f"Articles found: {len(articles)}\n")
for a in articles:
    print(f"  [{a['source']}] {a['title']}")

In [ ]:
#3. Market Agent (Obinna)
from pathlib import Path
import sys
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root))
from IPython.display import SVG, display
from agents.market_agent import MarketAgent
from tools.market_visualization import save_price_chart

ticker = "AAPL"  # Change this to another listed stock symbol.
market = MarketAgent().analyze(ticker, period="3mo")
print(f"{market['ticker']} as of {market['as_of']}")
print(market['summary'])
print(f"Current price: ${market['current_price']:,.2f}")
print(f"50-day average: {market['ma_50']}")
print(f"200-day average: {market['ma_200']}")

chart_path = save_price_chart(market, repo_root / "market_price_chart.svg")
display(SVG(filename=str(chart_path)))

---
## Workflow 1: Prompt Chaining

In [ ]:
from workflows.prompt_chaining import run_prompt_chain

chain_result = run_prompt_chain(TICKER, lookback_days=7)

print(f"Articles ingested:    {chain_result['articles_ingested']}")
print(f"After deduplication:  {chain_result['articles_after_dedup']}")
print(f"\nKey facts:")
for fact in chain_result['key_facts']:
    print(f"  - {fact}")

In [ ]:
print("News Summary:\n")
print(chain_result['summary'])

---
## Workflow 2: Routing
Clasifies a piece of content and routes it to the correct specialist.

In [ ]:
import os
import anthropic
from workflows.routing import classify_route

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

samples = [
    "Apple reported quarterly EPS of $1.64, beating estimates by $0.08. Revenue grew 5% YoY.",
    "Analyst upgrades AAPL to Buy with $220 price target citing strong services momentum.",
    "AAPL is trading above its 50-day moving average on above-average volume after a 3% breakout.",
]

for content in samples:
    route = classify_route(content, client)
    print(f"Route: {route:10s} | {content[:80]}")

---
## Workflow 3: Evaluator-Optimizer
Takes a draft analysis, scores it, refines it. Loops until quality passes threshold.

In [ ]:
from workflows.evaluator_optimizer import run_evaluator_optimizer

draft = chain_result['summary']  # Use the news summary as the draft to refine

eval_result = run_evaluator_optimizer(
    draft=draft,
    threshold=8,
    max_iterations=3
)

print(f"Iterations: {eval_result['iterations']}\n")
for h in eval_result['history']:
    print(f"  Iteration {h['iteration']}: score {h['score']}/10")
    print(f"  Feedback: {h['feedback'][:120]}...\n")

In [ ]:
print("Final refined draft:\n")
print(eval_result['final_draft'])

In [ ]:
#Research Agent (Viviana)

from agents.research_agent import ResearchAgent

agent = ResearchAgent()
report = agent.run(TICKER)

print("Research Plan:")
for i, step in enumerate(report['steps'], 1):
    print(f"  {i}. {step}")

In [ ]:
print("Market Snapshot:")
snap = report['market_snapshot']
print(f"  Price:    ${snap['price']:,.2f} ({snap['change_pct']:+.2f}%)")
print(f"  52W range: ${snap['week_52_low']} – ${snap['week_52_high']}")

print("\nFinancials Snapshot:")
fin = report['financials_snapshot']
print(f"  Revenue:    {fin['revenue']}")
print(f"  Net income: {fin['net_income']}")
print(f"  P/E ratio:  {fin['pe_ratio']}")
print(f"  Sector:     {fin['sector']}")

In [ ]:
print("Investment Analysis:\n")
print(report['analysis'])

In [ ]:
reflection = agent.reflect(report)
print("Self-Critique:\n")
print(reflection)